<a href="https://colab.research.google.com/github/EddyferO/Smart-Glass-Medical-Assistant/blob/main/notebooks/medgemma_in-context.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Load libraries

In [1]:
!pip install -q -U transformers bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 152.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.5 MB/s eta 0:00:00


## Download data

In [2]:
!wget -nc -q "https://drive.google.com/uc?export=download&id=1w0_XjEl4gCrDWxFnsJHrhsVEGTiF3Ayh" -O trainingData.zip
!unzip -q trainingData.zip

## Collect drug data

In [3]:
import json
import glob
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Data Loading Logic
def load_drug_data(directory):
    all_data = []
    file_paths = glob.glob(os.path.join(directory, "*.json"))
    for path in file_paths:
        with open(path, 'r') as f:
            try:
                item = json.load(f)
                all_data.append(item)
            except: continue
    return all_data

drug_data = load_drug_data("/content/trainingFilesConverted/")

In [4]:
drug_data[0]

{'drug': 'NitroMist',
 'text': '2 DOSAGE AND ADMINISTRATION\n\n At the onset of an attack, one metered spray or two metered sprays should be administered on or under the tongue. A spray may be repeated approximately every 5\xa0minutes as needed ( 2 ). \n Maximum of 3\xa0metered sprays are recommended within a 15-minute period. If chest pain persists after a total of 3\xa0sprays, prompt medical attention is recommended ( 2 ). \n May be used prophylactically 5\xa0minutes to 10\xa0minutes before engaging in activities that might precipitate an acute attack ( 2 ). \n\n2.1 Recommended Dosage\n\n At the onset of an attack, one metered spray or two metered sprays should be administered on or under the tongue. A spray may be repeated approximately every 5\xa0minutes as needed. If two sprays are used initially, the patient may only administer one more spray after waiting 5\xa0minutes. No more than 3\xa0metered sprays are recommended within a 15-minute period. If chest pain persists after a tota

## Get model and tokenizer

In [5]:
model_id = "google/medgemma-4b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

## Build in-context prompt

In [6]:
from IPython.display import display, Markdown

def run_in_context_extraction(target_drug, target_text, shots=2):
    # Select examples for the context
    context_examples = [ex for ex in drug_data if ex['drug'] != target_drug][:shots]

    # Build the prompt string
    prompt = "SYSTEM: You are a medical informatics expert. Extract drug-drug interactions into a structured JSON list.\n\n"

    for i, ex in enumerate(context_examples):
        prompt += f"### EXAMPLE {i+1}\n"
        prompt += f"DRUG: {ex['drug']}\n"
        prompt += f"TEXT: {ex['text'][:600]}...\n" # Truncated for token safety
        prompt += f"RESULT: {json.dumps(ex['interactions'])}\n\n"

    prompt += f"### TASK\nDRUG: {target_drug}\nTEXT:\n{target_text}\n\nRESULT:"

    # Visual
    display(Markdown("---"))
    display(Markdown(f"**The following text is being sent to the model:**\n\n```text\n{prompt}\n```"))

    # Execute
    formatted_input = f"<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n[" # Force JSON start
    inputs = tokenizer(formatted_input, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            temperature=0.01,
            do_sample=False
        )

    # Extract and format response
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    final_json = "[" + response

    display(Markdown(f"```json\n{final_json}\n```"))
    return final_json

## Sample it

In [7]:
# INPUT YOUR TEST CASE HERE
my_test_drug = "Warfarin" # @param {type: "string"}
my_test_text = "Warfarin metabolism is inhibited by fluconazole, leading to increased  prothrombin time. Concomitant administration of rifampin may decrease  warfarin effectiveness by inducing CYP enzymes. Patients should avoid  large amounts of leafy green vegetables (Vitamin K)." # @param {type: "string"}

# Execute
result = run_in_context_extraction(my_test_drug, my_test_text)

---

**The following text is being sent to the model:**

```text
SYSTEM: You are a medical informatics expert. Extract drug-drug interactions into a structured JSON list.

### EXAMPLE 1
DRUG: NitroMist
TEXT: 2 DOSAGE AND ADMINISTRATION

 At the onset of an attack, one metered spray or two metered sprays should be administered on or under the tongue. A spray may be repeated approximately every 5 minutes as needed ( 2 ). 
 Maximum of 3 metered sprays are recommended within a 15-minute period. If chest pain persists after a total of 3 sprays, prompt medical attention is recommended ( 2 ). 
 May be used prophylactically 5 minutes to 10 minutes before engaging in activities that might precipitate an acute attack ( 2 ). 

2.1 Recommended Dosage

 At the onset of an attack, one metered spray or two meter...
RESULT: [{"@type": "Pharmacodynamic interaction", "@precipitant": "pde5 inhibitors", "@precipitantCode": "N0000175599", "@effect": "45007003: Low blood pressure (disorder)"}, {"@type": "Unspecified interaction", "@precipitant": "pde5 inhibitors", "@precipitantCode": "N0000175599"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "sildenafil", "@precipitantCode": "N0000022115", "@effect": "45007003: Low blood pressure (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "tadalafil", "@precipitantCode": "N0000010536", "@effect": "45007003: Low blood pressure (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "vardenafil", "@precipitantCode": "N0000148820", "@effect": "45007003: Low blood pressure (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "ergotamine", "@precipitantCode": "N0000006354", "@effect": "51510002: Ergotism (disorder)"}, {"@type": "Pharmacokinetic interaction", "@precipitant": "ergotamine", "@precipitantCode": "N0000006354", "@effect": "C54357"}, {"@type": "Unspecified interaction", "@precipitant": "selective inhibitor of cyclic guanosine monophosphate (cgmp)-specific phosphodiesterase type5 (pde5)", "@precipitantCode": "NO MAP"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "heparin", "@precipitantCode": "N0000006341", "@effect": "76612001:  Hypercoagulability state (finding)"}, {"@type": "Unspecified interaction", "@precipitant": "heparin", "@precipitantCode": "N0000006341"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "aspirin", "@precipitantCode": "N0000006582", "@effect": " 422773005: Hemodynamic instability (finding)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "aspirin", "@precipitantCode": "N0000006582", "@effect": "91081006: Vasodilatation (finding)"}, {"@type": "Pharmacokinetic interaction", "@precipitant": "aspirin", "@precipitantCode": "N0000006582", "@effect": "C54355"}, {"@type": "Pharmacokinetic interaction", "@precipitant": "aspirin", "@precipitantCode": "N0000006582", "@effect": "C54602"}, {"@type": "Pharmacokinetic interaction", "@precipitant": "aspirin", "@precipitantCode": "N0000006582", "@effect": "C54605"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "labetolol", "@precipitantCode": "N0000006446", "@effect": "45007003: Low blood pressure (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "tissue-type plasminogen activator ( t-pa )", "@precipitantCode": "N0000007047", "@effect": "NO MAP"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "calcium channel blockers", "@precipitantCode": "N0000029119", "@effect": "28651003: Orthostatic hypotension (disorder)"}, {"@type": "Pharmacokinetic interaction", "@precipitant": "dihydroergotamine", "@precipitantCode": "N0000005941", "@effect": "C54357"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "antihypertensive drugs", "@precipitantCode": "N0000029427", "@effect": "45007003: Low blood pressure (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "beta-adrenergic blockers", "@precipitantCode": "N0000175556", "@effect": "45007003: Low blood pressure (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "nitrates", "@precipitantCode": "N0000007647", "@effect": "45007003: Low blood pressure (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "antihypertensives", "@precipitantCode": "N0000178477", "@effect": "45007003: Low blood pressure (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "tissue-type plasminogen activator (t-pa)", "@precipitantCode": "N0000007047", "@effect": "NO MAP"}, {"@type": "Pharmacokinetic interaction", "@precipitant": "t-pa", "@precipitantCode": "N0000007047", "@effect": "C54358"}, {"@type": "Unspecified interaction", "@precipitant": "t-pa", "@precipitantCode": "N0000007047"}]

### EXAMPLE 2
DRUG: edarbi
TEXT: WARNING: FETAL TOXICITY 

 When pregnancy is detected, discontinue Edarbi as soon as possible [see Warnings and Precautions (5.1) ]. 

 Drugs that act directly on the renin-angiotensin system can cause injury and death to the developing fetus [see 

 Warnings and Precautions (5.1) 
 ]. 

 WARNING: FETAL TOXICITY 
 See full prescribing information for complete boxed warning. 

 • 
 When pregnancy is detected, discontinue Edarbi as soon as possible. ( 5.1 ) 

 • 
 Drugs that act directly on the renin-angiotensin system can cause injury and death to the developing fetus. ( 5.1 )2 DOSAGE AND ADMIN...
RESULT: [{"@type": "Pharmacodynamic interaction", "@precipitant": "aliskiren", "@precipitantCode": "N0000176084", "@effect": " 14140009: Hyperkalemia (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "aliskiren", "@precipitantCode": "N0000176084", "@effect": " 14669001: Acute renal failure syndrome (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "aliskiren", "@precipitantCode": "N0000176084", "@effect": " 39539005: Abnormal renal function (finding)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "aliskiren", "@precipitantCode": "N0000176084", "@effect": "45007003: Low blood pressure (disorder)"}, {"@type": "Unspecified interaction", "@precipitant": "aliskiren", "@precipitantCode": "N0000176084"}, {"@type": "Unspecified interaction", "@precipitant": "agents that affect the ras", "@precipitantCode": "NO MAP"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "ace inhibitors", "@precipitantCode": "N0000029130", "@effect": " 14140009: Hyperkalemia (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "ace inhibitors", "@precipitantCode": "N0000029130", "@effect": " 14669001: Acute renal failure syndrome (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "ace inhibitors", "@precipitantCode": "N0000029130", "@effect": " 39539005: Abnormal renal function (finding)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "ace inhibitors", "@precipitantCode": "N0000029130", "@effect": "45007003: Low blood pressure (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "nsaids", "@precipitantCode": "N0000175722", "@effect": " 14669001: Acute renal failure syndrome (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "nsaids", "@precipitantCode": "N0000175722", "@effect": "314983004: Deteriorating renal function (finding)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "nsaids", "@precipitantCode": "N0000175722", "@effect": "NO MAP"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "selective cox-2 inhibitors", "@precipitantCode": "n0000008288", "@effect": " 14669001: Acute renal failure syndrome (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "selective cox-2 inhibitors", "@precipitantCode": "n0000008288", "@effect": "314983004: Deteriorating renal function (finding)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "selective cox-2 inhibitors", "@precipitantCode": "n0000008288", "@effect": "NO MAP"}, {"@type": "Unspecified interaction", "@precipitant": "nsaid", "@precipitantCode": "N0000175722"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "cox-2 inhibitors", "@precipitantCode": "n0000008288", "@effect": " 14669001: Acute renal failure syndrome (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "cox-2 inhibitors", "@precipitantCode": "n0000008288", "@effect": "314983004: Deteriorating renal function (finding)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "non-steroidal anti-inflammatory agents", "@precipitantCode": "n0000175722", "@effect": " 14669001: Acute renal failure syndrome (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "non-steroidal anti-inflammatory agents", "@precipitantCode": "n0000175722", "@effect": "314983004: Deteriorating renal function (finding)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "selective cyclooxygenase-2 inhibitors", "@precipitantCode": "n0000008288", "@effect": " 14669001: Acute renal failure syndrome (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "selective cyclooxygenase-2 inhibitors", "@precipitantCode": "n0000008288", "@effect": "314983004: Deteriorating renal function (finding)"}]

### TASK
DRUG: Warfarin
TEXT:
Warfarin metabolism is inhibited by fluconazole, leading to increased  prothrombin time. Concomitant administration of rifampin may decrease  warfarin effectiveness by inducing CYP enzymes. Patients should avoid  large amounts of leafy green vegetables (Vitamin K).

RESULT:
```

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


```json
[
  {
    "@type": "Pharmacokinetic interaction",
    "@precipitant": "fluconazole",
    "@precipitantCode": "N0000006354",
    "@effect": "C54357"
  },
  {
    "@type": "Pharmacokinetic interaction",
    "@precipitant": "rifampin",
    "@precipitantCode": "N0000006582",
    "@effect": "C54357"
  },
  {
    "@type": "Pharmacodynamic interaction",
    "@precipitant": "leafy green vegetables",
    "@precipitantCode": "N0000006582",
    "@effect": "45007003: Low blood pressure (disorder)"
  }
]

```

## API drug information testing

### openFDA API

In [8]:
import requests
import json
from IPython.display import display, Markdown

def fetch_fda_interactions(drug_name):
    """
    Retrieves Section 7 (Drug Interactions) from the openFDA API.
    """
    # API endpoint
    base_url = "https://api.fda.gov/drug/label.json"

    # Build query
    params = {
        'search': f'openfda.brand_name:"{drug_name}" openfda.generic_name:"{drug_name}"',
        'limit': 1
    }

    try:
        response = requests.get(base_url, params=params)
        response.raise_for_status()
        data = response.json()

        if 'results' in data:
            label = data['results'][0]
            # 'drug_interactions' is the standard field for Section 7
            raw_text = label.get('drug_interactions', ["Section 7 not found in this label."])[0]

            return {
                "drug": drug_name.upper(),
                "text": raw_text,
                "source": "openFDA API"
            }
        return None
    except Exception as e:
        display(Markdown(f"**Error fetching {drug_name}:** {str(e)}"))
        return None

# Test
fetch_drug = "Epidiolex" # @param {type: "string"}
fetched = fetch_fda_interactions(fetch_drug)
if fetched:
    display(Markdown(f"### API Data: {fetched['drug']}"))
    display(Markdown(f"**Raw Section 7 Snippet:**\n> {fetched['text'][:500]}..."))

### API Data: EPIDIOLEX

**Raw Section 7 Snippet:**
> 7 DRUG INTERACTIONS • Strong inducer of CYP3A4 or CYP2C19: Consider dose increase of EPIDIOLEX. ( 7.1 ) • Consider a dose reduction of substrates of CYP1A2, CYP2C8, UGT1A9, and orally administered P-gp substrates. ( 7.2 ) • A lower starting dose of orally administered everolimus is recommended. ( 7.2 ) • Consider dose modification of CYP2B6 or CYP2C19 substrates. ( 7.2 ) 7.1 Effect of Other Drugs on EPIDIOLEX Strong CYP3A4 or CYP2C19 Inducers Concomitant use with a strong CYP3A4 and CYP2C19 indu...

### DailyMed API

In [9]:
import requests
from xml.etree import ElementTree
from IPython.display import display, Markdown

def fetch_setId(drug_name):
    """
    Retrieves the SET ID for the first drug with the given name from the DailyMed API.
    """
    # API endpoint
    endpoint = "https://dailymed.nlm.nih.gov/dailymed/services/v2/spls.json"

    # Build query
    params = {
        'drug_name': drug_name
    }

    try:
        response = requests.get(endpoint, params=params)
        response.raise_for_status()
        data = response.json()

        if 'data' in data:
            print(f"Found {len(data['data'])} results for {drug_name}.")
            print(f"Choosing index 0: {data['data'][0]['title']}")
            return data['data'][0]['setid']
        return None
    except Exception as e:
        display(Markdown(f"**Error fetching SET ID for {drug_name}:** {str(e)}"))
        return None

def fetch_dailyMed_interactions(drug_name):
    """
    Retrieves Section 7 (Drug Interactions) from the DailyMed API.
    """
    # Get the SET ID for the drug name
    setId = fetch_setId(drug_name)
    if setId == None:
      return None

    # API endpoint
    endpoint = f"https://dailymed.nlm.nih.gov/dailymed/services/v2/spls/{setId}.xml"

    try:
        response = requests.get(endpoint)
        response.raise_for_status()
        tree = ElementTree.fromstring(response.content)

        section = tree.find(".//{urn:hl7-org:v3}code[@code='34073-7']/..")

        if section is not None:
            return {
                "drug": drug_name.upper(),
                "text": "".join(section.itertext()).strip(),
                "source": "DailyMed API"
            }
        return None
    except Exception as e:
        display(Markdown(f"**Error fetching {drug_name}:** {str(e)}"))
        return None

# Test
fetch_drug = "Epidiolex" # @param {type: "string"}
fetched = fetch_dailyMed_interactions(fetch_drug)
if fetched:
    display(Markdown(f"### API Data: {fetched['drug']}"))
    display(Markdown(f"**Raw Section 7 Snippet:**\n> {fetched['text'][:1000]}..."))

Found 1 results for Epidiolex.
Choosing index 0: EPIDIOLEX (CANNABIDIOL) SOLUTION [JAZZ PHARMACEUTICALS, INC.]


### API Data: EPIDIOLEX

**Raw Section 7 Snippet:**
> 7  DRUG INTERACTIONS
               
               
                  
                     
                        
                           
                              •Strong inducer of CYP3A4 or CYP2C19:  Consider dose increase of EPIDIOLEX. (7.1)
                           
                              •Consider a dose reduction of substrates of CYP1A2, CYP2C8, UGT1A9, and orally administered P-gp substrates. (7.2)
                           
                              •A lower starting dose of orally administered everolimus is recommended. (7.2)
                           
                              •Consider dose modification of CYP2B6 or CYP2C19 substrates. (7.2)
                        
                     
                  
               
               
                  
                     
                     
                     7.1  Effect of Other Drugs on EPIDIOLEX
                     
                        
                           Strong CYP...

In [11]:
def test_new_drug_with_medgemma(drug_name, dailyMed=True):
    # Collect the FDA label
    collected_data = fetch_dailyMed_interactions(drug_name) if dailyMed else fetch_fda_interactions(drug_name)

    if not collected_data:
        display(Markdown(f"Could not find a label for **{drug_name}**"))
        return

    # Pass the collected text to our extraction engine
    display(Markdown(f"###Testing on: **{drug_name}**"))

    final_json = run_in_context_extraction(
        target_drug=collected_data['drug'],
        target_text=collected_data['text']
    )

    return final_json

# Try a drug that wasn't in the training set to see if it understands the pattern
test_drug = "Amlodipine" # @param {type: "string"}
use_dailyMed = True # @param {type: "boolean"}
test_case = test_new_drug_with_medgemma(test_drug)

Found 100 results for Amlodipine.
Choosing index 0: AMLODIPINE AND ATORVASTATIN TABLET, FILM COATED [BRYANT RANCH PREPACK]


###Testing on: **Amlodipine**

---

**The following text is being sent to the model:**

```text
SYSTEM: You are a medical informatics expert. Extract drug-drug interactions into a structured JSON list.

### EXAMPLE 1
DRUG: NitroMist
TEXT: 2 DOSAGE AND ADMINISTRATION

 At the onset of an attack, one metered spray or two metered sprays should be administered on or under the tongue. A spray may be repeated approximately every 5 minutes as needed ( 2 ). 
 Maximum of 3 metered sprays are recommended within a 15-minute period. If chest pain persists after a total of 3 sprays, prompt medical attention is recommended ( 2 ). 
 May be used prophylactically 5 minutes to 10 minutes before engaging in activities that might precipitate an acute attack ( 2 ). 

2.1 Recommended Dosage

 At the onset of an attack, one metered spray or two meter...
RESULT: [{"@type": "Pharmacodynamic interaction", "@precipitant": "pde5 inhibitors", "@precipitantCode": "N0000175599", "@effect": "45007003: Low blood pressure (disorder)"}, {"@type": "Unspecified interaction", "@precipitant": "pde5 inhibitors", "@precipitantCode": "N0000175599"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "sildenafil", "@precipitantCode": "N0000022115", "@effect": "45007003: Low blood pressure (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "tadalafil", "@precipitantCode": "N0000010536", "@effect": "45007003: Low blood pressure (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "vardenafil", "@precipitantCode": "N0000148820", "@effect": "45007003: Low blood pressure (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "ergotamine", "@precipitantCode": "N0000006354", "@effect": "51510002: Ergotism (disorder)"}, {"@type": "Pharmacokinetic interaction", "@precipitant": "ergotamine", "@precipitantCode": "N0000006354", "@effect": "C54357"}, {"@type": "Unspecified interaction", "@precipitant": "selective inhibitor of cyclic guanosine monophosphate (cgmp)-specific phosphodiesterase type5 (pde5)", "@precipitantCode": "NO MAP"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "heparin", "@precipitantCode": "N0000006341", "@effect": "76612001:  Hypercoagulability state (finding)"}, {"@type": "Unspecified interaction", "@precipitant": "heparin", "@precipitantCode": "N0000006341"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "aspirin", "@precipitantCode": "N0000006582", "@effect": " 422773005: Hemodynamic instability (finding)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "aspirin", "@precipitantCode": "N0000006582", "@effect": "91081006: Vasodilatation (finding)"}, {"@type": "Pharmacokinetic interaction", "@precipitant": "aspirin", "@precipitantCode": "N0000006582", "@effect": "C54355"}, {"@type": "Pharmacokinetic interaction", "@precipitant": "aspirin", "@precipitantCode": "N0000006582", "@effect": "C54602"}, {"@type": "Pharmacokinetic interaction", "@precipitant": "aspirin", "@precipitantCode": "N0000006582", "@effect": "C54605"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "labetolol", "@precipitantCode": "N0000006446", "@effect": "45007003: Low blood pressure (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "tissue-type plasminogen activator ( t-pa )", "@precipitantCode": "N0000007047", "@effect": "NO MAP"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "calcium channel blockers", "@precipitantCode": "N0000029119", "@effect": "28651003: Orthostatic hypotension (disorder)"}, {"@type": "Pharmacokinetic interaction", "@precipitant": "dihydroergotamine", "@precipitantCode": "N0000005941", "@effect": "C54357"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "antihypertensive drugs", "@precipitantCode": "N0000029427", "@effect": "45007003: Low blood pressure (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "beta-adrenergic blockers", "@precipitantCode": "N0000175556", "@effect": "45007003: Low blood pressure (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "nitrates", "@precipitantCode": "N0000007647", "@effect": "45007003: Low blood pressure (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "antihypertensives", "@precipitantCode": "N0000178477", "@effect": "45007003: Low blood pressure (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "tissue-type plasminogen activator (t-pa)", "@precipitantCode": "N0000007047", "@effect": "NO MAP"}, {"@type": "Pharmacokinetic interaction", "@precipitant": "t-pa", "@precipitantCode": "N0000007047", "@effect": "C54358"}, {"@type": "Unspecified interaction", "@precipitant": "t-pa", "@precipitantCode": "N0000007047"}]

### EXAMPLE 2
DRUG: edarbi
TEXT: WARNING: FETAL TOXICITY 

 When pregnancy is detected, discontinue Edarbi as soon as possible [see Warnings and Precautions (5.1) ]. 

 Drugs that act directly on the renin-angiotensin system can cause injury and death to the developing fetus [see 

 Warnings and Precautions (5.1) 
 ]. 

 WARNING: FETAL TOXICITY 
 See full prescribing information for complete boxed warning. 

 • 
 When pregnancy is detected, discontinue Edarbi as soon as possible. ( 5.1 ) 

 • 
 Drugs that act directly on the renin-angiotensin system can cause injury and death to the developing fetus. ( 5.1 )2 DOSAGE AND ADMIN...
RESULT: [{"@type": "Pharmacodynamic interaction", "@precipitant": "aliskiren", "@precipitantCode": "N0000176084", "@effect": " 14140009: Hyperkalemia (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "aliskiren", "@precipitantCode": "N0000176084", "@effect": " 14669001: Acute renal failure syndrome (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "aliskiren", "@precipitantCode": "N0000176084", "@effect": " 39539005: Abnormal renal function (finding)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "aliskiren", "@precipitantCode": "N0000176084", "@effect": "45007003: Low blood pressure (disorder)"}, {"@type": "Unspecified interaction", "@precipitant": "aliskiren", "@precipitantCode": "N0000176084"}, {"@type": "Unspecified interaction", "@precipitant": "agents that affect the ras", "@precipitantCode": "NO MAP"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "ace inhibitors", "@precipitantCode": "N0000029130", "@effect": " 14140009: Hyperkalemia (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "ace inhibitors", "@precipitantCode": "N0000029130", "@effect": " 14669001: Acute renal failure syndrome (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "ace inhibitors", "@precipitantCode": "N0000029130", "@effect": " 39539005: Abnormal renal function (finding)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "ace inhibitors", "@precipitantCode": "N0000029130", "@effect": "45007003: Low blood pressure (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "nsaids", "@precipitantCode": "N0000175722", "@effect": " 14669001: Acute renal failure syndrome (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "nsaids", "@precipitantCode": "N0000175722", "@effect": "314983004: Deteriorating renal function (finding)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "nsaids", "@precipitantCode": "N0000175722", "@effect": "NO MAP"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "selective cox-2 inhibitors", "@precipitantCode": "n0000008288", "@effect": " 14669001: Acute renal failure syndrome (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "selective cox-2 inhibitors", "@precipitantCode": "n0000008288", "@effect": "314983004: Deteriorating renal function (finding)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "selective cox-2 inhibitors", "@precipitantCode": "n0000008288", "@effect": "NO MAP"}, {"@type": "Unspecified interaction", "@precipitant": "nsaid", "@precipitantCode": "N0000175722"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "cox-2 inhibitors", "@precipitantCode": "n0000008288", "@effect": " 14669001: Acute renal failure syndrome (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "cox-2 inhibitors", "@precipitantCode": "n0000008288", "@effect": "314983004: Deteriorating renal function (finding)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "non-steroidal anti-inflammatory agents", "@precipitantCode": "n0000175722", "@effect": " 14669001: Acute renal failure syndrome (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "non-steroidal anti-inflammatory agents", "@precipitantCode": "n0000175722", "@effect": "314983004: Deteriorating renal function (finding)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "selective cyclooxygenase-2 inhibitors", "@precipitantCode": "n0000008288", "@effect": " 14669001: Acute renal failure syndrome (disorder)"}, {"@type": "Pharmacodynamic interaction", "@precipitant": "selective cyclooxygenase-2 inhibitors", "@precipitantCode": "n0000008288", "@effect": "314983004: Deteriorating renal function (finding)"}]

### TASK
DRUG: AMLODIPINE
TEXT:
7  DRUG INTERACTIONS
               
                  Data from a drug-drug interaction study involving 10 mg of amlodipine and 80 mg of atorvastatin in healthy subjects indicate that the pharmacokinetics of amlodipine are not altered when the drugs are co-administered. The effect of amlodipine on the pharmacokinetics of atorvastatin showed no effect on the Cmax: 91% (90% confidence interval: 80 to 103%), but the AUC of atorvastatin increased by 18% (90% confidence interval: 109 to 127%) in the presence of amlodipine, which is not clinically meaningful.
                  No drug interaction studies have been conducted with amlodipine and atorvastatin and other drugs, although studies have been conducted in the individual amlodipine and atorvastatin components, as described below:
                  
                     Amlodipine
                  
               
               
               
                  
                     
                        
                           See full prescribing information for details regarding concomitant use of amlodipine and atorvastatin tablets with other drugs or grapefruit juice that increase the risk of myopathy and rhabdomyolysis (2.5, 7.3).
                           Rifampin: May reduce atorvastatin plasma concentrations. Administer simultaneously with atorvastatin (7.4)
                           Oral Contraceptives: May increase plasma levels of norethindrone and ethinyl estradiol; consider this effect when selecting an oral contraceptive (7.5).
                           Digoxin: May increase digoxin plasma levels; monitor patients appropriately (7.5).
                        
                        
                     
                  
               
               
                  
                     
                     
                     7.1 Impact of Other Drugs on Amlodipine
                     
                     
                        
                           CYP3A Inhibitors
                        
                        Co-administration with CYP3A inhibitors (moderate and strong) results in increased systemic exposure to amlodipine and may require dose reduction. Monitor for symptoms of hypotension and edema when amlodipine is co-administered with CYP3A inhibitors to determine the need for dose adjustment [see Clinical Pharmacology (12.3)].
                        
                           CYP3A Inducers
                        
                        No information is available on the quantitative effects of CYP3A inducers on amlodipine. Blood pressure should be closely monitored when amlodipine is co-administered with CYP3A inducers.
                        
                           Sildenafil
                        
                        Monitor for hypotension when sildenafil is co-administered with amlodipine [see Clinical Pharmacology (12.2)].
                     
                     
                  
               
               
                  
                     
                     
                     
                        7.2 Impact of Amlodipine on Other Drugs
                     
                     
                        
                           Immunosuppressants
                        
                        Amlodipine may increase the systemic exposure of cyclosporine or tacrolimus when co-administered. Frequent monitoring of trough blood levels of cyclosporine and tacrolimus is recommended and adjust the dose when appropriate [see Clinical Pharmacology (12.3)].
                        
                           Atorvastatin
                        
                        
                     
                     
                  
               
               
                  
                     
                     
                     7.3 Drug Interactions that may Increase the Risk of Myopathy and Rhabdomyolysis with Atorvastatin
                     
                        Atorvastatin is a substrate of CYP3A4 and transporters (e.g., OATP1B1/1B3, P-gp, or BCRP). Atorvastatin plasma levels can be significantly increased with concomitant administration of inhibitors of CYP3A4 and transporters. Table 3 includes a list of drugs that may increase exposure to atorvastatin and may increase the risk of myopathy and rhabdomyolysis when used concomitantly and instructions for preventing or managing them [see Warnings and Precautions (5.1) and Clinical Pharmacology (12.3)].
                        
                           Table 3. Drug Interactions that may Increase the Risk of Myopathy and Rhabdomyolysis with Atorvastatin
                        
                        
                           
                              
                                 
                                    Cyclosporine or Gemfibrozil
                                 
                              
                              
                                 
                                    
                                     Clinical Impact: 
                                  Atorvastatin plasma levels were significantly increased with concomitant administration of atorvastatin and cyclosporine, an inhibitor of CYP3A4 and OATP1B1 [see Clinical Pharmacology (12.3)]. Gemfibrozil may cause myopathy when given alone. The risk of myopathy and rhabdomyolysis is increased with concomitant use of cyclosporine or gemfibrozil with atorvastatin .
                              
                              
                                  Intervention: 
                                  Concomitant use of cyclosporine or gemfibrozil with atorvastatin is not recommended. 
                              
                              
                                 
                                    Anti-Viral Medications
                                    
                                 
                              
                              
                                  Clinical Impact: 
                                  Atorvastatin plasma levels were significantly increased with concomitant administration of atorvastatin with many anti-viral medications, which are inhibitors of CYP3A4 and/or transporters (e.g., BCRP, OATP1B1/1B3, P-gp, MRP2, and/or OAT2) [see Clinical Pharmacology (12.3)]. Cases of myopathy and rhabdomyolysis have been reported with concomitant use of ledipasvir plus sofosbuvir with atorvastatin. 
                              
                              
                                  Intervention: 
                                 
                                    
                                       Concomitant use of tipranavir plus ritonavir or glecaprevir plus pibrentasvir with atorvastatin is not recommended.
                                       In patients taking lopinavir plus ritonavir, or simeprevir, consider the risk/benefit of concomitant use with atorvastatin.
                                       In patients taking saquinavir plus ritonavir, darunavir plus ritonavir, fosamprenavir, fosamprenavir plus ritonavir, elbasvir plus grazoprevir or letermovir do not exceed atorvastatin 20 mg.
                                       In patients taking nelfinavir, do not exceed atorvastatin 40 mg [see Dosage and Administration (2)].
                                       Consider the risk/benefit of concomitant use of ledipasvir plus sofosbuvir with atorvastatin.
                                       Monitor all patients for signs and symptoms of myopathy particularly during initiation of therapy and during upward dose titration of either drug.
                                    
                                 
                              
                              
                                  Examples: 
                                  Tipranavir plus ritonavir, glecaprevir plus pibrentasvir, lopinavir plus ritonavir, simeprevir, saquinavir plus ritonavir, darunavir plus ritonavir, fosamprenavir, fosamprenavir plus ritonavir, elbasvir plus grazoprevir, letermovir, nelfinavir, and ledipasvir plus sofosbuvir .
                              
                              
                                 
                                    Select Azole Antifungals or Macrolide Antibiotics
                                    
                                 
                              
                              
                                 
                                     Clinical Impact: 
                                  Atorvastatin plasma levels were significantly increased with concomitant administration of atorvastatin with select azole antifungals or macrolide antibiotics, due to inhibition of CYP3A4 and/or transporters [see Clinical Pharmacology (12.3)]. 
                              
                              
                                 
                                     Intervention: 
                                  In patients taking clarithromycin or itraconazole, do not exceed atorvastatin 20 mg [see Dosage and Administration (2)]. Consider the risk/benefit of concomitant use of other azole antifungals or macrolide antibiotics with atorvastatin. Monitor all patients for signs and symptoms of myopathy particularly during initiation of therapy and during upward dose titration of either drug. 
                              
                              
                                  Examples: 
                                  Erythromycin, clarithromycin, itraconazole, ketoconazole, posaconazole, and voriconazole. 
                              
                              
                                 
                                    Niacin
                                    
                                 
                              
                              
                                  Clinical Impact: 
                                  Cases of myopathy and rhabdomyolysis have been observed with concomitant use of lipid modifying dosages of niacin (≥1 gram/day niacin) with atorvastatin. 
                              
                              
                                 
                                     Intervention: 
                                  Consider if the benefit of using lipid modifying dosages of niacin concomitantly with atorvastatin outweighs the increased risk of myopathy and rhabdomyolysis. If concomitant use is decided, monitor patients for signs and symptoms of myopathy particularly during initiation of therapy and during upward dose titration of either drug. 
                              
                              
                                 
                                    Fibrates (other than Gemfibrozil)
                                    
                                 
                              
                              
                                  Clinical Impact: 
                                  Fibrates may cause myopathy when given alone. The risk of myopathy and rhabdomyolysis is increased with concomitant use of fibrates with atorvastatin. 
                              
                              
                                 
                                     Intervention: 
                                  Consider if the benefit of using fibrates concomitantly with atorvastatin outweighs the increased risk of myopathy and rhabdomyolysis. If concomitant use is decided, monitor patients for signs and symptoms of myopathy particularly during initiation of therapy and during upward dose titration of either drug. 
                              
                              
                                 
                                    Colchicine
                                    
                                 
                              
                              
                                  Clinical Impact: 
                                  Cases of myopathy and rhabdomyolysis have been reported with concomitant use of colchicine with atorvastatin. 
                              
                              
                                 
                                     Intervention: 
                                  Consider the risk/benefit of concomitant use of colchicine with atorvastatin. If concomitant use is decided, monitor patients for signs and symptoms of myopathy particularly during initiation of therapy and during upward dose titration of either drug. 
                              
                              
                                 
                                    Grapefruit Juice
                                    
                                 
                              
                              
                                  Clinical Impact: 
                                  Grapefruit juice consumption, especially excessive consumption, more than 1.2 liters/daily can raise the plasma levels of atorvastatin and may increase the risk of myopathy and rhabdomyolysis. 
                              
                              
                                  Intervention: 
                                  Avoid intake of large quantities of grapefruit juice, more than 1.2 liters daily, when taking atorvastatin. 
                              
                           
                        
                        
                     
                     
                  
               
               
                  
                     
                     
                     7.4 Drug Interactions that may Decrease Exposure to Atorvastatin
                     
                        Table 4 presents drug interactions that may decrease exposure to atorvastatin and instructions for preventing or managing them.
                        
                           Table 4. Drug Interactions that may Decrease Exposure to Atorvastatin
                        
                        
                           
                              
                                 
                                    Rifampin
                                 
                                 
                              
                              
                                 
                                    Clinical Impact:
                                 
                                 Concomitant administration of atorvastatin with rifampin, an inducer of cytochrome P450 3A4 and inhibitor of OATP1B1, can lead to variable reductions in plasma concentrations of atorvastatin. Due to the dual interaction mechanism of rifampin, delayed administration of atorvastatin after administration of rifampin has been associated with a significant reduction in atorvastatin plasma concentrations.
                              
                              
                                 
                                    Intervention:
                                 
                                 Administer atorvastatin and rifampin simultaneously.
                              
                           
                        
                        
                     
                     
                  
               
               
                  
                     
                     
                     7.5 Atorvastatin Effects on Other Drugs
                     
                     
                        Table 5 presents atorvastatin’s effect on other drugs and instructions for preventing or managing them.
                        
                           Table 5. Atorvastatin Effects on Other Drugs
                        
                        
                           
                              
                                 
                                    Oral Contraceptives
                                 
                                 
                              
                              
                                 
                                    Clinical Impact:
                                 
                                 Co-administration of atorvastatin and an oral contraceptive increased plasma concentrations of norethindrone and ethinyl estradiol [see Clinical Pharmacology (12.3)].
                              
                              
                                 
                                    Intervention:
                                 
                                 Consider this when selecting an oral contraceptive for patients taking atorvastatin.
                              
                              
                                 
                                    Digoxin
                                 
                                 
                              
                              
                                 
                                    Clinical Impact:
                                 
                                 When multiple doses of atorvastatin and digoxin were co-administered, steady state plasma digoxin concentrations increased [see Clinical Pharmacology (12.3)].
                              
                              
                                 
                                    Intervention:
                                 
                                 Monitor patients taking digoxin appropriately.

RESULT:
```

```json
[
  {
    "@type": "Pharmacodynamic interaction",
    "@precipitant": "aliskiren",
    "@precipitantCode": "N0000176084",
    "@effect": "14140009: Hyperkalemia (disorder)"
  },
  {
    "@type": "Pharmacodynamic interaction",
    "@precipitant": "aliskiren",
    "@precipitantCode": "N0000176084",
    "@effect": "14669001: Acute renal failure syndrome (disorder)"
  },
  {
    "@type": "Pharmacodynamic interaction",
    "@precipitant": "aliskiren",
    "@precipitantCode": "N0000176084",
    "@effect": "39539005: Abnormal renal function (finding)"
  },
  {
    "@type": "Pharmacodynamic interaction",
    "@precipitant": "aliskiren",
    "@precipitantCode": "N0000176084",
    "@effect": "45007003: Low blood pressure (disorder)"
  },
  {
    "@type": "Unspecified interaction",
    "@precipitant": "aliskiren",
    "@precipitantCode": "N0000176084"
  },
  {
    "@type": "Pharmacodynamic interaction",
    "@precipitant": "aliskiren",
    "@precipitantCode": "N0000176084",
    "@effect": "14140009: Hyperkalemia (disorder)"
  },
  {
    "@type": "Pharmacodynamic interaction",
    "@precipitant": "aliskiren",
    "@precipitantCode": "N0000176084",
    "@effect": "14669001: Acute renal failure syndrome (disorder)"
  },
  {
    "@type": "Pharmacodynamic interaction",
    "@precipitant": "aliskiren",
    "@precipitantCode": "N0000176084",
    "@effect": "39539005: Abnormal renal function (finding)"
  },
  {
    "@type": "Pharmacodynamic interaction",
    "@precipitant": "aliskiren",
    "@precipitantCode": "N0000176084",
    "@effect": "45007003: Low blood pressure (disorder)"
  },
  {
    "@type": "Unspecified interaction",
    "@precipitant": "aliskiren",
    "@precipitantCode": "N0000176084"
  },
  {
    "@type": "Pharmacodynamic interaction",
    "@precipitant": "aliskiren",
    "@precipitantCode": "N0000176084",
    "@effect": "14140009: Hyperkalemia (disorder)"
  },
  {
    "@type": "Pharmacodynamic interaction",
    "@precipitant": "aliskiren",
    "@precipitantCode": "N0000176084",
    "@effect": "14669001: Acute renal failure syndrome (disorder)"
  },
  {
    "@type": "Pharmacodynamic interaction",
    "@precipitant": "aliskiren",
    "@precipitantCode": "N0000176084",
    "@effect": "39539005: Abnormal renal function (finding)"
  },
  {
    "@type": "Pharmacodynamic interaction",
    "@precipitant": "aliskiren",
    "@precipitantCode": "N0000176084",
    "@effect": "45007003: Low blood pressure (disorder)"
  },
  {
    "@type": "Unspecified interaction",
    "@precipitant": "aliskiren",
    "@precipitantCode": "N0000176084"
  },
  {
    "@type": "Pharmacodynamic interaction",
    "@precipitant": "aliskiren",
```